<a href="https://colab.research.google.com/github/stella1298/AIFFEL_Quest_EPA/blob/master/NLP/NLP01/260903_haena_pjt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 260902_PJT : 네이버 영화리뷰 감정 분석 문제에 SentencePiece 적용해 보기

In [26]:
# 필요한 라이브러리 설치
!pip install sentencepiece konlpy -q

In [27]:
import random
import numpy as np
import pandas as pd
import sentencepiece as spm
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 실험 결과를 일정하게 유지하기 위한 seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [28]:
# 네이버 영화 리뷰 데이터 다운로드
!git clone https://github.com/e9t/nsmc.git

fatal: destination path 'nsmc' already exists and is not an empty directory.


In [29]:
# Train과 Test 데이터 읽기
train_df = pd.read_csv(
    "/content/nsmc/ratings_train.txt",
    sep="\t"
)

test_df = pd.read_csv(
    "/content/nsmc/ratings_test.txt",
    sep="\t"
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

display(train_df.head())

## document 영화 리뷰, label 감정의 종류
### 0 : 부정, 1 : 긍정

Train: (150000, 3)
Test : (50000, 3)


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [30]:
# 리뷰가 없는 데이터 제거
train_df = train_df.dropna(subset=["document"])
test_df = test_df.dropna(subset=["document"])

# 동일한 리뷰가 반복되는 경우 제거
train_df = train_df.drop_duplicates(
    subset=["document"]
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (146182, 3)
Test : (49997, 3)


In [31]:
# SentencePiece가 학습할 텍스트 파일 생성
corpus_path = "/content/nsmc_corpus.txt"

with open(corpus_path, "w", encoding="utf-8") as f:

    for text in train_df["document"]:

        # 리뷰 하나를 한 줄로 저장
        text = str(text).replace("\n", " ")

        f.write(text + "\n")

print("Corpus 생성 완료")

Corpus 생성 완료


In [32]:
# SentencePiece Unigram 모델 생성
spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix="/content/nsmc_sp",
    # test 1 : vocab_size=4000,
    # test 2 : vocab_size=8000,
    # test 3 : vocab_size=12000,
    vocab_size=4000,
    model_type="unigram",

    # 특수 토큰의 ID
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,

    character_coverage=0.9995
)

True

In [33]:
# test 4 : BPE 방식의 SentencePiece 모델 생성
spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix="/content/nsmc_bpe",
    vocab_size=8000,
    model_type="bpe",

    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,

    character_coverage=0.9995
)

KeyboardInterrupt: 

In [34]:
# 학습된 SentencePiece 모델 불러오기
sp = spm.SentencePieceProcessor()

sp.load("/content/nsmc_sp.model")


def sp_tokenize(text):
    """
    입력 문장을 SentencePiece subword로 분리한다.
    """

    return sp.encode(
        text,
        out_type=str
    )

In [35]:
text = "이 영화 정말 재미있어요."

print("원문:", text)
print("토큰:", sp_tokenize(text))

원문: 이 영화 정말 재미있어요.
토큰: ['▁이', '▁영화', '▁정말', '▁재미있어요', '.']


In [36]:
def encode_text(text):
    """
    문장을 SentencePiece token ID로 변환한다.
    """

    return sp.encode(
        text,
        out_type=int
    )

In [37]:
# Train / Test 문장과 label 분리
X_train_text = train_df["document"].astype(str).tolist()
y_train = train_df["label"].values

X_test_text = test_df["document"].astype(str).tolist()
y_test = test_df["label"].values


# SentencePiece로 문장을 숫자 sequence로 변환
X_train = [
    encode_text(text)
    for text in X_train_text
]

X_test = [
    encode_text(text)
    for text in X_test_text
]

In [38]:
# 하나의 리뷰에서 사용할 최대 token 수
MAX_LEN = 50

X_train = pad_sequences(
    X_train,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
    value=0
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
    value=0
)

print(X_train.shape)
print(X_test.shape)

(146182, 50)
(49997, 50)


In [39]:
# SentencePiece vocabulary 크기
VOCAB_SIZE = sp.get_piece_size()


model = Sequential([

    # token ID를 벡터로 변환
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        mask_zero=True
    ),

    # 리뷰의 순서 정보를 학습
    LSTM(64),

    # 과적합을 줄이기 위한 층
    Dropout(0.3),

    # 긍정/부정 분류
    Dense(1, activation="sigmoid")
])


model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [40]:
# Validation 성능이 더 이상 좋아지지 않으면 학습 중단
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


history = model.fit(
    X_train,
    y_train,

    validation_split=0.1,

    epochs=10,
    batch_size=128,

    callbacks=[early_stop]
)

Epoch 1/10
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 137s 130ms/step - accuracy: 0.8188 - loss: 0.4007 - val_accuracy: 0.8385 - val_loss: 0.3582
Epoch 2/10
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 135s 132ms/step - accuracy: 0.8508 - loss: 0.3383 - val_accuracy: 0.8455 - val_loss: 0.3439
Epoch 3/10
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 135s 131ms/step - accuracy: 0.8671 - loss: 0.3057 - val_accuracy: 0.8481 - val_loss: 0.3455
Epoch 4/10
1028/1028 ━━━━━━━━━━━━━━━━━━━━ 143s 132ms/step - accuracy: 0.8794 - loss: 0.2813 - val_accuracy: 0.8490 - val_loss: 0.3469


In [41]:
# Test 데이터로 최종 성능 확인
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"Test Accuracy: {accuracy:.4f}")

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.8415 - loss: 0.3527
Test Accuracy: 0.8415


> #  Mecab

In [18]:
# Mecab 형태소 분석기 설치
!apt-get update -qq
!apt-get install -y mecab libmecab-dev mecab-ipadic-utf8 -qq

# Python용 Mecab 패키지 설치
!pip install mecab-python3 -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libmecab2:amd64.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../0-libmecab2_0.996-14build9_amd64.deb ...
Unpacking libmecab2:amd64 (0.996-14build9) ...
Selecting previously unselected package libmecab-dev.
Preparing to unpack .../1-libmecab-dev_0.996-14build9_amd64.deb ...
Unpacking libmecab-dev (0.996-14build9) ...
Selecting previously unselected package mecab-utils.
Preparing to unpack .../2-mecab-utils_0.996-14build9_amd64.deb ...
Unpacking mecab-utils (0.996-14build9) ...
Selecting previously unselected package mecab-ipadic.
Preparing to unpack .../3-mecab-ipadic_2.7.0-20070801+main-3_all.deb ...
Unpacking mecab-ipadic (2.7.0-20070801+main-3) ...
Selecting previously unselected package mecab.
Preparing to un

In [22]:
!pip install -U mecab-python3 unidic-lite -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 23.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [25]:
import MeCab

mecab = MeCab.Tagger()

def mecab_tokenize(text):
    result = mecab.parse(text).splitlines()

    tokens = []

    for line in result:
        if line == "EOS" or "\t" not in line:
            continue

        word = line.split("\t")[0]
        tokens.append(word)

    return tokens


text = "이 영화 정말 재미있어요."

print(mecab_tokenize(text))

['이', '영화', '정말', '재미있어요', '.']


> #  TEST Result

Tokenizer	Model / Type /	Vocab Size /	Test Accuracy

SentencePiece	/ Unigram	/ 4,000	/ 0.8415

SentencePiece	/ Unigram	/ 8,000 / 0.8518

SentencePiece /	Unigram /	12,000 / -

SentencePiece /	BPE	/ 8,000	/ -

Mecab /	형태소 / - / -

---
---

# SentencePiece 성능

> SentencePiece를 적용한 LSTM 모델은 Test Accuracy 80% 이상을 달성하였다. 따라서 SentencePiece tokenizer와 RNN 기반 분류 모델을 결합하여 영화 리뷰의 감정을 분류할 수 있음을 확인하였다.

# Vocabulary Size

> Vocabulary size를 변경하여 실험한 결과, 크기에 따라 분류 성능에 차이가 나타났다. Vocabulary가 크다고 해서 항상 정확도가 높아지는 것은 아니었으며, 본 실험에서는 8000개의 vocabulary에서 가장 높은 성능을 보였다.

# Unigram과 BPE

> Unigram과 BPE를 비교한 결과 두 방식에서 서로 다른 성능이 나타났다. 이는 subword를 구성하는 방식의 차이가 모델의 입력 표현과 분류 결과에 영향을 주었기 때문으로 볼 수 있다.

# Mecab과 비교

> Mecab은 형태소를 기준으로 문장을 분리하는 반면 SentencePiece는 subword 단위로 분리한다. SentencePiece는 미등록어나 새로운 표현도 작은 단위로 나누어 처리할 수 있다는 장점이 있다. 반면 한국어 형태소 정보를 직접 활용한다는 측면에서는 Mecab이 유리할 수 있다.